Script de extração dos dados via API

In [3]:
# Import das bibliotecas
import requests
import boto3
import os
import logging
import json
from botocore.client import Config
from botocore.exceptions import ClientError
from conf.settings import settings
from datetime import datetime


# Paths
GITHUB_URL = settings.github_url
MINIO_ENDPOINT = settings.minio_endpoint
BUCKET_NAME = settings.minio_bucket
BRONZE_DATA = settings.bronze_data
MINIO_ACCESS_KEY = settings.minio_access_key
MINIO_SECRET_KEY = settings.minio_secret_key

# Configuração de Log
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# Função de conexão do MiniO via boto3
def conexao_minio_client():
    return boto3.client(
        "s3",
        endpoint_url=MINIO_ENDPOINT,
        aws_access_key_id=MINIO_ACCESS_KEY,
        aws_secret_access_key=MINIO_SECRET_KEY,
        config=Config(signature_version="s3v4"),
        region_name="us-east-1",
    )

# Função para verificar se bucket existe, senão cria
def garantir_bucket(s3):
    try:
        s3.head_bucket(Bucket=BUCKET_NAME)
    except ClientError:
        print(f"Criando bucket: {BUCKET_NAME}")
        s3.create_bucket(Bucket=BUCKET_NAME)

# Função para extração dos dados via API
def extrair_dados():
    logging.info("Iniciando extração...")

    response = requests.get(GITHUB_URL, timeout=10)

    if response.status_code != 200:
        raise Exception("Erro ao acessar repositório da API")

    arquivos = response.json()
    s3 = conexao_minio_client()
    garantir_bucket(s3)
    
    # Partição por data e id de execução
    ingestion_date = datetime.utcnow().strftime("%Y-%m-%d")
    execution_id = datetime.utcnow().strftime("%Y%m%dT%H%M%S")

    arquivos_enviados = []

    for arquivo in arquivos:
        nome = arquivo["name"]
        download_url = arquivo["download_url"]
        
        chave = (
            f"{BRONZE_DATA}raw/"
            f"ingestion_date={ingestion_date}/"
            f"execution_id={execution_id}/"
            f"{nome}"
        )

        logging.info(f"Processando {nome}...")
        
        file_response = requests.get(download_url, timeout=10)
       
        # Verificar se o retorno é OK
        if file_response.status_code == 200:
            s3.put_object(
                Bucket=BUCKET_NAME,
                Key=chave,
                Body=file_response.content,
            )
            arquivos_enviados.append(nome)
        else:
            logging.warning(f"Erro ao baixar {nome}")
    
    logging.info(f"{len(arquivos_enviados)} arquivos enviados")
    
    # Metadados por execução na camada Bronze (Governaça de dados)
    metadata = {
        "execution_id": execution_id,
        "ingestion_date": ingestion_date,
        "total_files": len(arquivos_enviados),
        "status": "success"
    }
    
    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=f"{BRONZE_DATA}raw/ingestion_date={ingestion_date}/execution_id={execution_id}/_metadata.json",
        Body=json.dumps(metadata),
    )
    
    return arquivos_enviados
        

In [ ]:
# Extrair os dados para o Data lake
extrair_dados()

2026-03-03 00:50:38,299 - INFO - Iniciando extração...
2026-03-03 00:50:39,027 - INFO - Processando Aaron697_Brekke496_2fa15bc7-8866-461a-9000-f739e425860a.json...
2026-03-03 00:50:39,287 - INFO - Processando Aaron697_Stiedemann542_41166989-975d-4d17-b9de-17f94cb3eec1.json...
2026-03-03 00:50:39,543 - INFO - Processando Abby752_Kuvalis369_2b083021-e93f-4991-bf49-fd4f20060ef8.json...
2026-03-03 00:50:39,750 - INFO - Processando Abel832_Connelly992_29e51479-f742-4474-8f8e-d2607d5269f6.json...
2026-03-03 00:50:39,974 - INFO - Processando Abraham100_Heller342_262b819a-5193-404a-9787-b7f599358035.json...
2026-03-03 00:50:40,210 - INFO - Processando Adam631_Cronin387_aff8f143-2375-416f-901d-b0e4c73e3e58.json...
2026-03-03 00:50:40,443 - INFO - Processando Adam631_Shields502_9e2653fc-49e0-4b2e-86f8-e664bbe07be3.json...
2026-03-03 00:50:40,726 - INFO - Processando Adelaida985_Gulgowski816_86662c9c-fcc8-41a6-a2e8-61270bf8a3b0.json...
2026-03-03 00:50:40,984 - INFO - Processando Adolfo777_O'Keef

2026-03-03 00:50:55,817 - INFO - Processando April374_Gerhold939_e3c6cb9c-5f13-4d78-bc38-d7c005346389.json...
2026-03-03 00:50:56,032 - INFO - Processando Archie818_Barrows492_fef3d64f-82b0-47e0-be02-3317c4969826.json...
2026-03-03 00:50:56,435 - INFO - Processando Ardath226_Zboncak558_cb96223c-c32b-45be-8aa7-36795883a060.json...
2026-03-03 00:50:56,664 - INFO - Processando Ardelia466_Larson43_68b95ae3-8732-4948-9345-f3d490bc4ae5.json...
2026-03-03 00:51:06,727 - INFO - Processando Arica110_Herman763_c08a8771-4cfb-458f-a248-d4dad06a6e17.json...
2026-03-03 00:50:56,814 - INFO - Processando Ariel183_Friesen796_1ae87f34-f0c9-4312-91cd-349dd9b790b7.json...
2026-03-03 00:50:57,041 - INFO - Processando Ariel183_Turner526_6edf069d-ebad-4c90-8192-119fb32c84c1.json...
2026-03-03 00:50:57,274 - INFO - Processando Arielle168_Marvin195_c76c71f4-7193-4d78-9e36-c99011243b72.json...
2026-03-03 00:50:57,513 - INFO - Processando Armand155_Mills423_9e36e8d9-b864-453f-a2ef-6eac2df7f080.json...
2026-03-03

2026-03-03 00:51:13,396 - INFO - Processando Bud153_Lang846_83cd3265-e2b8-4a84-95fb-491738d4fd75.json...
2026-03-03 00:51:13,622 - INFO - Processando Buffy238_Schumm995_ccb86a73-115d-4f69-8c13-50083e549438.json...
2026-03-03 00:51:13,901 - INFO - Processando Buffy238_Wiza601_a99b82af-6654-4101-8091-cea62bae7826.json...
2026-03-03 00:51:14,192 - INFO - Processando Bunny174_Sauer652_3038f107-f43e-4f8d-bee3-0dd51236ac73.json...
2026-03-03 00:51:14,444 - INFO - Processando Burl285_Torp761_2b4109b0-0a81-409d-ac15-ecc25bb91175.json...
2026-03-03 00:51:14,693 - INFO - Processando Burma963_Oberbrunner298_f42c7e4c-fc13-4aa7-9d58-352d31ba9105.json...
2026-03-03 00:51:15,030 - INFO - Processando Buster609_Durgan499_d8c908da-de6b-41da-8abf-96b27df2518f.json...
2026-03-03 00:51:15,341 - INFO - Processando Buster609_King743_53d82273-242b-438d-8056-adc9dc3f3078.json...
2026-03-03 00:51:15,586 - INFO - Processando Byron202_Vandervort697_5cb3f714-91f4-4b9f-bd58-dc64d8dd3a04.json...
2026-03-03 00:51:15,

2026-03-03 00:51:30,820 - INFO - Processando Chuck784_Nader710_669cd39f-c5fc-41c3-aff3-1ed7be7d40a7.json...
2026-03-03 00:51:31,086 - INFO - Processando Chung121_Koch169_8027e27d-405f-490c-a946-91d1b40c8fd0.json...
2026-03-03 00:51:31,362 - INFO - Processando Cicely661_Parker433_75d3758c-f568-448f-bb24-9699dc5732e1.json...
2026-03-03 00:51:31,588 - INFO - Processando Clair921_Weimann465_614b9e91-dcbd-4db4-9302-1d7fecac2bed.json...
2026-03-03 00:51:31,805 - INFO - Processando Clarence5_Abernathy524_3f31e949-d916-476d-bf37-f6874ab7b1c1.json...
2026-03-03 00:51:41,782 - INFO - Processando Claudia969_González124_6e4e4311-55f3-4c2d-8576-d4cb63f121b6.json...
2026-03-03 00:51:31,852 - INFO - Processando Claudine313_Schuppe920_a9df73f7-dd25-4b5b-b924-938edabfb461.json...
2026-03-03 00:51:32,092 - INFO - Processando Claudio955_Contreras711_f5b5e542-7999-45ed-bd61-3e21856ab21c.json...
